In [1]:
import pymysql
from neo4j import GraphDatabase, basic_auth
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
from datetime import datetime, date
from decimal import Decimal

# --- 구성 (Configuration) ---
MAX_WORKERS = 5
BATCH_SIZE = 5000

MYSQL_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "dev",
    "password": "pwd",
    "database": "orders",
}

NEO4J_CONFIG = {
    "uri": "bolt://localhost:7687",
    "username": "neo4j",
    "password": "gustjs21@", # 실제 Neo4j 비밀번호로 교체하세요!
    "database": 'test02', # 현재 사용 중인 데이터베이스 이름
}

# --- 로깅 설정 (Logging Setup) ---
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

file_handler = logging.FileHandler('migration.log')
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# --- 헬퍼 함수 (Helper Functions) ---

def clean_property_name(name):
    cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
    if cleaned_name and not re.match(r'^[a-zA-Z_]', cleaned_name[0]):
        cleaned_name = '_' + cleaned_name
    if not cleaned_name or cleaned_name == '_':
        return f"prop_{name}"
    return cleaned_name

def custom_json_serializer(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Decimal):
        return float(obj)
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

def clear_neo4j_database(driver):
    logger.info("Neo4j 데이터베이스를 초기화 중입니다...")
    try:
        with driver.session(database=NEO4J_CONFIG["database"]) as session:
            session.run("CALL apoc.periodic.iterate('MATCH (n) RETURN n', 'DETACH DELETE n', {batchSize: 10000, parallel: true})")
        logger.info("Neo4j 데이터베이스 초기화 완료.")
    except Exception as e:
        logger.error(f"Neo4j 데이터베이스 초기화 중 오류 발생: {e}", exc_info=True)


def create_constraints(driver, node_label, primary_key_column):
    neo4j_pk_column = clean_property_name(primary_key_column)
    try:
        with driver.session(database=NEO4J_CONFIG["database"]) as session:
            cypher_query = f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{node_label}) REQUIRE n.{neo4j_pk_column} IS UNIQUE"
            session.run(cypher_query)
            logger.info(f"'{node_label}' 레이블의 '{neo4j_pk_column}' 속성에 대한 고유성 제약 조건 생성 완료.")
    except Exception as e:
        logger.warning(f"'{node_label}' 레이블의 '{neo4j_pk_column}' 속성에 대한 제약 조건 생성 실패: {e}. (쿼리: {cypher_query})", exc_info=True)

def create_nodes_batch(driver, label, records, primary_key_col_name):
    logger.debug(f"Neo4j에 '{label}' 노드 {len(records)}개 생성/병합 중...")
    try:
        with driver.session(database=NEO4J_CONFIG["database"]) as session:
            cypher = (
                f"UNWIND $rows AS row\n"
                f"MERGE (n:{label} {{ {primary_key_col_name}: row.{primary_key_col_name} }})\n"
                f"SET n = row"
            )
            session.run(cypher, rows=records)
        logger.debug(f"'{label}' 노드 {len(records)}개 생성/병합 완료.")
    except Exception as e:
        logger.error(f"'{label}' 노드 생성/병합 중 오류 발생 (배치 크기: {len(records)}): {e}", exc_info=True)


def migrate_nodes_batched(mysql_conn, driver, table_name, node_label, primary_key_column_name):
    offset = 0
    total_processed = 0
    cursor = mysql_conn.cursor(pymysql.cursors.DictCursor)
    futures = []
    
    logger.info(f"테이블 '{table_name}'에서 '{node_label}' 노드로 마이그레이션 시작...")

    cursor.execute(f"SHOW COLUMNS FROM `{table_name}` LIKE 'deletedAt'")
    has_deleted_at = cursor.fetchone() is not None

    where_clause = " WHERE deletedAt IS NULL" if has_deleted_at else ""

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        while True:
            cursor.execute(f"SELECT * FROM `{table_name}`{where_clause} LIMIT {BATCH_SIZE} OFFSET {offset}")
            rows = cursor.fetchall()
            if not rows:
                break

            processed_rows = []
            for row in rows:
                cleaned_row = {}
                for k, v in row.items():
                    cleaned_k = clean_property_name(k)
                    if isinstance(v, Decimal):
                        cleaned_row[cleaned_k] = float(v)
                    elif isinstance(v, (datetime, date)):
                        cleaned_row[cleaned_k] = v.isoformat()
                    elif isinstance(v, (dict, list)):
                        cleaned_row[cleaned_k] = json.dumps(v, default=custom_json_serializer)
                    else:
                        cleaned_row[cleaned_k] = v
                processed_rows.append(cleaned_row)

            futures.append(executor.submit(create_nodes_batch, driver, node_label, processed_rows, clean_property_name(primary_key_column_name)))
            offset += BATCH_SIZE
            logger.info(f"테이블 '{table_name}'에서 {len(rows)}개의 레코드 배치 제출. 총 제출된 레코드: {offset}")

        for future in as_completed(futures):
            try:
                future.result() 
                total_processed += BATCH_SIZE 
            except Exception as exc:
                logger.error(f"'{node_label}' 노드 생성 중 치명적인 오류 발생: {exc}", exc_info=True)
    logger.info(f"테이블 '{table_name}'의 '{node_label}' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): {total_processed}")


def migrate_relationship(mysql_conn, driver, from_table, from_label, mysql_from_key, to_label, mysql_to_key, relationship_type):
    offset = 0
    total_relationships_created = 0
    cursor = mysql_conn.cursor(pymysql.cursors.DictCursor)

    neo4j_from_key = clean_property_name(mysql_from_key)
    neo4j_to_key = clean_property_name(mysql_to_key)

    logger.info(f"관계 마이그레이션 시작: {from_label}.{mysql_from_key} -[{relationship_type}]-> {to_label}.{mysql_to_key}")

    cursor.execute(f"SHOW COLUMNS FROM `{from_table}` LIKE 'deletedAt'")
    has_deleted_at = cursor.fetchone() is not None

    where_clause = f" WHERE `{mysql_from_key}` IS NOT NULL AND `{mysql_to_key}` IS NOT NULL"
    if has_deleted_at:
        where_clause += " AND deletedAt IS NULL"

    with driver.session(database=NEO4J_CONFIG["database"]) as session:
        while True:
            # MySQL에서 관계 생성을 위한 ID 쌍 가져오기
            query = f"SELECT `{mysql_from_key}`, `{mysql_to_key}` FROM `{from_table}`{where_clause} LIMIT {BATCH_SIZE} OFFSET {offset}"
            cursor.execute(query)
            rows = cursor.fetchall()

            if not rows:
                break

            cypher_query = f"""
            UNWIND $batch AS rel
            MATCH (a:{from_label} {{ {neo4j_from_key}: rel.`{mysql_from_key}` }})
            MATCH (b:{to_label} {{ {neo4j_to_key}: rel.`{mysql_to_key}` }})
            MERGE (a)-[:{relationship_type}]->(b)
            """
            try:
                session.run(cypher_query, batch=rows)
                total_relationships_created += len(rows)
                logger.info(f"관계 배치 처리: {len(rows)}개 ({from_label}-[:{relationship_type}]->{to_label}). 총 {total_relationships_created}개.")
            except Exception as e:
                logger.error(f"관계 생성 중 오류 발생 ({from_label}-[:{relationship_type}]->{to_label}, 배치 크기: {len(rows)}): {e}", exc_info=True)

            offset += BATCH_SIZE
    logger.info(f"관계 마이그레이션 완료: {from_label}-[:{relationship_type}]->{to_label}. 총 {total_relationships_created}개의 관계 생성.")


def main():
    logger.info("🔥 마이그레이션 프로세스 시작 🔥")
    mysql_conn = None
    neo4j_driver = None
    try:
        logger.info("MySQL 데이터베이스에 연결 중...")
        mysql_conn = pymysql.connect(**MYSQL_CONFIG, charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)
        logger.info("MySQL 연결 성공.")

        logger.info("Neo4j 데이터베이스에 연결 중...")
        neo4j_driver = GraphDatabase.driver(
            NEO4J_CONFIG["uri"], auth=basic_auth(NEO4J_CONFIG["username"], NEO4J_CONFIG["password"])
        )
        neo4j_driver.verify_connectivity()
        logger.info("Neo4j 연결 성공.")

        clear_neo4j_database(neo4j_driver)

        table_node_config = {
            "categories": ["Category", "id"],
            "countries": ["Country", "alpha2code"], # Country의 PK는 그대로 'alpha2code'로 유지
            "coupon_usages": ["CouponUsage", "id"],
            "coupons": ["Coupon", "id"],
            "customers": ["Customer", "id"],
            "food_courts": ["FoodCourt", "id"],
            "grouped_texts": ["GroupedText", "id"],
            "inventory_items": ["InventoryItem", "id"],
            "inventory_transactions": ["InventoryTransaction", "id"],
            "locales": ["Locale", "code"],
            "menu_option_choice_to_menu_options": ["MenuOptionChoiceToMenuOption", "id"],
            "menu_option_choices": ["MenuOptionChoice", "id"],
            "menu_option_to_menus": ["MenuOptionToMenu", "id"],
            "menu_options": ["MenuOption", "id"],
            "menu_to_inventory_items": ["MenuToInventoryItem", "id"],
            "menus": ["Menu", "id"],
            "order_items": ["OrderItem", "id"],
            "orders": ["Order", "id"],
            "payment_option_enums": ["PaymentOptionEnum", "paymentOptionEnum"],
            "payments": ["Payment", "id"],
            "refund_items": ["RefundItem", "id"],
            "refunds": ["Refund", "id"],
            "store_attributes": ["StoreAttribute", "id"],
            "store_meta_advert_insights": ["StoreMetaAdvertInsight", "id"],
            "store_meta_adverts": ["StoreMetaAdvert", "id"],
            "store_tables": ["StoreTable", "id"],
            "stores": ["Store", "id"],
        }

        logger.info("--- 노드 마이그레이션 및 제약 조건 생성 시작 ---")
        for mysql_table, (neo4j_label, mysql_pk_column_name) in table_node_config.items():
            create_constraints(neo4j_driver, neo4j_label, mysql_pk_column_name)
            migrate_nodes_batched(mysql_conn, neo4j_driver, mysql_table, neo4j_label, mysql_pk_column_name)
        logger.info("--- 노드 마이그레이션 및 제약 조건 생성 완료 ---")

        logger.info("--- 관계 마이그레이션 시작 ---")
        relationships_to_migrate = [
            # categories 테이블 (Category 노드)
            ("categories", "Category", "groupedTextId", "GroupedText", "id", "HAS_TEXT_DESCRIPTION"),
            ("categories", "Category", "storeId", "Store", "id", "BELONONGS_TO_STORE"),

            # coupon_usages 테이블 (CouponUsage 노드)
            ("coupon_usages", "CouponUsage", "couponId", "Coupon", "id", "USED_COUPON"),
            ("coupon_usages", "CouponUsage", "customerId", "Customer", "id", "USED_BY"),
            ("coupon_usages", "CouponUsage", "orderId", "Order", "id", "APPLIED_TO_ORDER"),

            # coupons 테이블 (Coupon 노드)
            ("coupons", "Coupon", "storeId", "Store", "id", "ISSUED_BY"),

            # food_courts 테이블과 countries 테이블 간의 관계는 제외됨.
            # ("food_courts", "FoodCourt", "countryCode", "Country", "alpha2code", "LOCATED_IN_COUNTRY"), # <<< 이 라인 제거

            # grouped_texts 테이블 (GroupedText 노드)
            ("grouped_texts", "GroupedText", "originalLocaleCode", "Locale", "code", "HAS_ORIGINAL_LOCALE"),

            # inventory_items 테이블 (InventoryItem 노드)
            ("inventory_items", "InventoryItem", "storeId", "Store", "id", "STOCKED_IN"),

            # inventory_transactions 테이블 (InventoryTransaction 노드)
            ("inventory_transactions", "InventoryTransaction", "inventoryItemId", "InventoryItem", "id", "AFFECTS_ITEM"),
            ("inventory_transactions", "InventoryTransaction", "storeId", "Store", "id", "OCCURRED_AT_STORE"),

            # menu_option_choice_to_menu_options 테이블 (중간 테이블: MenuOptionChoiceToMenuOption 노드)
            ("menu_option_choice_to_menu_options", "MenuOptionChoiceToMenuOption", "menuOptionChoiceId", "MenuOptionChoice", "id", "HAS_CHOICE_REF"),
            ("menu_option_choice_to_menu_options", "MenuOptionChoiceToMenuOption", "menuOptionId", "MenuOption", "id", "HAS_OPTION_REF"),
            
            # menu_option_choices 테이블 (MenuOptionChoice 노드)
            ("menu_option_choices", "MenuOptionChoice", "storeId", "Store", "id", "BELONGS_TO_STORE_MOC"),
            ("menu_option_choices", "MenuOptionChoice", "groupedTextId", "GroupedText", "id", "HAS_NAME_TEXT_MOC"),

            # menu_option_to_menus 테이블 (중간 테이블: MenuOptionToMenu 노드)
            ("menu_option_to_menus", "MenuOptionToMenu", "menuId", "Menu", "id", "ASSOCIATED_WITH_MENU"),
            ("menu_option_to_menus", "MenuOptionToMenu", "menuOptionId", "MenuOption", "id", "ASSOCIATED_WITH_OPTION"),
            
            # menu_options 테이블 (MenuOption 노드)
            ("menu_options", "MenuOption", "storeId", "Store", "id", "BELONGS_TO_STORE_MO"),
            ("menu_options", "MenuOption", "groupedTextId", "GroupedText", "id", "HAS_NAME_TEXT_MO"),

            # menu_to_inventory_items 테이블 (중간 테이블: MenuToInventoryItem 노드)
            ("menu_to_inventory_items", "MenuToInventoryItem", "menuId", "Menu", "id", "LINKED_TO_MENU"),
            ("menu_to_inventory_items", "MenuToInventoryItem", "inventoryItemId", "InventoryItem", "id", "LINKED_TO_INVENTORY_ITEM"),
            
            # menus 테이블 (Menu 노드)
            ("menus", "Menu", "storeId", "Store", "id", "OFFERED_BY"),
            ("menus", "Menu", "categoryId", "Category", "id", "BELONGS_TO_CATEGORY"),
            ("menus", "Menu", "groupedTextId", "GroupedText", "id", "HAS_NAME_TEXT_MENU"),

            # order_items 테이블 (OrderItem 노드)
            ("order_items", "OrderItem", "orderId", "Order", "id", "PART_OF"),
            ("order_items", "OrderItem", "menuId", "Menu", "id", "REFERENCES"),

            # orders 테이블 (Order 노드)
            ("orders", "Order", "customerId", "Customer", "id", "PLACED_BY"),
            ("orders", "Order", "storeId", "Store", "id", "PLACED_AT"),
            ("orders", "Order", "paymentOptionEnum", "PaymentOptionEnum", "paymentOptionEnum", "USES_PAYMENT_OPTION"),
            ("orders", "Order", "storeTableId", "StoreTable", "id", "RESERVED_TABLE"),
            ("orders", "Order", "paymentId", "Payment", "id", "HAS_PAYMENT"),

            # payments 테이블 (Payment 노드)
            ("payments", "Payment", "orderId", "Order", "id", "FOR_ORDER"),
            ("payments", "Payment", "customerId", "Customer", "id", "MADE_BY_CUSTOMER"),
            ("payments", "Payment", "storeId", "Store", "id", "MADE_AT_STORE"),

            # refund_items 테이블 (RefundItem 노드)
            ("refund_items", "RefundItem", "refundId", "Refund", "id", "IS_PART_OF_REFUND"),
            ("refund_items", "RefundItem", "orderItemId", "OrderItem", "id", "REFUNDS_ITEM"),

            # refunds 테이블 (Refund 노드)
            ("refunds", "Refund", "orderId", "Order", "id", "FOR_ORDER_REFUND"),
            ("refunds", "Refund", "paymentId", "Payment", "id", "RELATED_TO_PAYMENT_REFUND"),
            ("refunds", "Refund", "storeId", "Store", "id", "INITIATED_BY_STORE"),

            # store_attributes 테이블 (StoreAttribute 노드)
            ("store_attributes", "StoreAttribute", "storeId", "Store", "id", "FOR_STORE"),

            # store_meta_advert_insights 테이블 (StoreMetaAdvertInsight 노드)
            ("store_meta_advert_insights", "StoreMetaAdvertInsight", "storeMetaAdvertId", "StoreMetaAdvert", "id", "HAS_INSIGHT"),

            # store_meta_adverts 테이블 (StoreMetaAdvert 노드)
            ("store_meta_adverts", "StoreMetaAdvert", "storeId", "Store", "id", "FOR_STORE_ADVERTISEMENT"),

            # stores 테이블 (Store 노드)
            ("stores", "Store", "foodCourtId", "FoodCourt", "id", "LOCATED_IN_FOODCOURT"), # 이 관계는 그대로 유지
        ]

        for rel_info in relationships_to_migrate:
            from_table, from_label, mysql_from_key, to_label, mysql_to_key, relationship_type = rel_info
            migrate_relationship(mysql_conn, neo4j_driver, from_table, from_label, mysql_from_key, to_label, mysql_to_key, relationship_type)

        logger.info("--- 관계 마이그레이션 완료 ---")
        logger.info("✅ 마이그레이션 프로세스 전체 완료 ✅")

    except pymysql.Error as e:
        logger.error(f"MySQL 연결 또는 쿼리 오류: {e}", exc_info=True)
    except Exception as e:
        logger.error(f"예상치 못한 오류 발생: {e}", exc_info=True)
    finally:
        if mysql_conn:
            mysql_conn.close()
            logger.info("MySQL 연결 종료.")
        if neo4j_driver:
            neo4j_driver.close()
            logger.info("Neo4j 연결 종료.")

if __name__ == "__main__":
    main()

2025-07-09 17:51:28,140 - INFO - 🔥 마이그레이션 프로세스 시작 🔥
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\logging\__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
UnicodeEncodeError: 'cp949' codec can't encode character '\U0001f525' in position 44: illegal multibyte sequence
Call stack:
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\ipykernel\kernelapp.py", line 

2025-07-09 17:51:59,371 - INFO - 'InventoryTransaction' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:51:59,373 - INFO - 테이블 'inventory_transactions'에서 'InventoryTransaction' 노드로 마이그레이션 시작...
2025-07-09 17:51:59,397 - INFO - 테이블 'inventory_transactions'에서 186개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:51:59,413 - INFO - 테이블 'inventory_transactions'의 'InventoryTransaction' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:51:59,416 - INFO - 'Locale' 레이블의 'code' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:51:59,418 - INFO - 테이블 'locales'에서 'Locale' 노드로 마이그레이션 시작...
2025-07-09 17:51:59,441 - INFO - 테이블 'locales'에서 183개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:51:59,464 - INFO - 테이블 'locales'의 'Locale' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:51:59,468 - INFO - 'MenuOptionChoiceToMenuOption' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:51:59,470 - INFO - 테이블 'menu_option_choice_to_menu_options'에서 'MenuOptionChoiceToMenuOption' 노드로 마이그레이션 시작...
2025-07-09 17:51:59,557 - INFO - 테이블 

2025-07-09 17:53:37,782 - INFO - 테이블 'refund_items'에서 1137개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:53:37,891 - INFO - 테이블 'refund_items'의 'RefundItem' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:53:37,897 - INFO - 'Refund' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:53:37,899 - INFO - 테이블 'refunds'에서 'Refund' 노드로 마이그레이션 시작...
2025-07-09 17:53:37,987 - INFO - 테이블 'refunds'에서 943개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:53:38,166 - INFO - 테이블 'refunds'의 'Refund' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:53:38,171 - INFO - 'StoreAttribute' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:53:38,173 - INFO - 테이블 'store_attributes'에서 'StoreAttribute' 노드로 마이그레이션 시작...
2025-07-09 17:53:38,285 - INFO - 테이블 'store_attributes'에서 1453개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:53:38,429 - INFO - 테이블 'store_attributes'의 'StoreAttribute' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:53:38,433 - INFO - 'StoreMetaAdvertInsight' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:

In [1]:
# main.py
import pymysql
from neo4j import GraphDatabase
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import math

# --- Configuration ---
MYSQL_HOST = '127.0.0.1'
MYSQL_PORT = 3306
MYSQL_USER = 'dev'
MYSQL_PASSWORD = 'pwd'
MYSQL_DB = 'orders' # The database name from your schema dump

NEO4J_URI = 'bolt://localhost:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = 'gustjs21@'
NEO4J_DATABASE = 'neo4j' # Target Neo4j database

BATCH_SIZE = 5000  # Number of nodes/relationships to create per batch
MAX_WORKERS = 8 # Adjust based on your CPU cores and I/O capacity

# --- Logging Setup ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(threadName)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('migration.log')
    ]
)
logger = logging.getLogger(__name__)

# --- Database Connections ---
def get_mysql_connection():
    """Establishes and returns a MySQL connection."""
    try:
        conn = pymysql.connect(
            host=MYSQL_HOST,
            port=MYSQL_PORT,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=MYSQL_DB,
            cursorclass=pymysql.cursors.DictCursor
        )
        logger.info("Successfully connected to MySQL database.")
        return conn
    except pymysql.Error as e:
        logger.error(f"Error connecting to MySQL: {e}")
        raise

def get_neo4j_driver():
    """Establishes and returns a Neo4j driver."""
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        driver.verify_connectivity()
        logger.info("Successfully connected to Neo4j database.")
        return driver
    except Exception as e:
        logger.error(f"Error connecting to Neo4j: {e}")
        raise

# --- Migration Functions ---

def create_nodes_batched(tx, label, properties_list):
    """
    Creates nodes in Neo4j using UNWIND for batching.
    """
    query = f"""
    UNWIND $properties_list AS properties
    CREATE (n:{label})
    SET n = properties
    """
    tx.run(query, properties_list=properties_list)

def migrate_table_to_nodes(mysql_conn, neo4j_driver, table_name, id_column='id'):
    """
    Migrates data from a MySQL table to Neo4j nodes with a given label,
    using multithreading and batching.
    """
    logger.info(f"Starting migration for table: {table_name} to nodes with label: {table_name.capitalize()}")
    total_rows = 0
    try:
        with mysql_conn.cursor() as cursor:
            cursor.execute(f"SELECT COUNT(*) AS count FROM `{table_name}` WHERE deletedAt IS NULL")
            total_rows = cursor.fetchone()['count']
            logger.info(f"Found {total_rows} active records in MySQL table `{table_name}`.")

            if total_rows == 0:
                logger.info(f"No active records to migrate for table `{table_name}`. Skipping node creation.")
                return

            offset = 0
            futures = []
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                while offset < total_rows:
                    fetch_query = f"SELECT * FROM `{table_name}` WHERE deletedAt IS NULL LIMIT {BATCH_SIZE} OFFSET {offset}"
                    cursor.execute(fetch_query)
                    rows = cursor.fetchall()
                    if not rows:
                        break

                    properties_list = []
                    for row in rows:
                        props = {}
                        for k, v in row.items():
                            # Convert any non-serializable types (e.g., decimals, datetimes) to string
                            if isinstance(v, (pymysql.times.Timestamp, pymysql.times.Date)):
                                props[k] = v.isoformat()
                            elif isinstance(v, (float, int)): # Handle decimals by converting to float
                                props[k] = float(v)
                            elif v is None:
                                pass # Skip None values to avoid creating properties with null
                            else:
                                props[k] = v
                        properties_list.append(props)

                    futures.append(executor.submit(
                        neo4j_driver.execute_write,
                        create_nodes_batched,
                        table_name.capitalize(), # Use capitalized table name as label
                        properties_list
                    ))
                    offset += len(rows)

                for i, future in enumerate(as_completed(futures)):
                    try:
                        future.result()
                        logger.info(f"Batch {i+1} for table '{table_name}' nodes completed.")
                    except Exception as e:
                        logger.error(f"Error creating nodes for table '{table_name}' in a batch: {e}")
            logger.info(f"Finished migrating table `{table_name}` to nodes.")

    except pymysql.Error as e:
        logger.error(f"MySQL error during migration of table `{table_name}`: {e}")
    except Exception as e:
        logger.error(f"An unexpected error occurred during node migration for table `{table_name}`: {e}")

def create_relationships_apoc(neo4j_driver, start_label, start_id_column, end_label, end_id_column, relationship_type, relationship_id_column):
    """
    Creates relationships natively in Neo4j using apoc.periodic.iterate.
    Assumes IDs are direct matches between nodes.
    """
    logger.info(f"Creating relationships of type :{relationship_type} between {start_label} and {end_label}...")

    # Ensure the target Neo4j database is used for the APOC call
    apoc_query = f"""
    CALL apoc.periodic.iterate(
        "MATCH (start_node:{start_label}) WHERE start_node.{start_id_column} IS NOT NULL RETURN start_node",
        "MATCH (end_node:{end_label}) WHERE end_node.{end_id_column} = start_node.{relationship_id_column} CREATE (start_node)-[:{relationship_type}]->(end_node)",
        {{batchSize: {BATCH_SIZE}, parallel: true, iterateList: true}}
    )
    """
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            session.run(apoc_query)
        logger.info(f"Successfully created relationships of type :{relationship_type}.")
    except Exception as e:
        logger.error(f"Error creating relationships of type :{relationship_type}: {e}")


def get_table_schema(mysql_conn, table_name):
    """Fetches the schema for a given MySQL table, including foreign key information."""
    schema = {}
    try:
        with mysql_conn.cursor() as cursor:
            cursor.execute(f"SHOW COLUMNS FROM `{table_name}`")
            schema['columns'] = cursor.fetchall()

            # Get foreign key constraints
            fk_query = f"""
            SELECT
                COLUMN_NAME,
                REFERENCED_TABLE_NAME,
                REFERENCED_COLUMN_NAME,
                CONSTRAINT_NAME
            FROM
                INFORMATION_SCHEMA.KEY_COLUMN_USAGE
            WHERE
                TABLE_SCHEMA = '{MYSQL_DB}' AND
                TABLE_NAME = '{table_name}' AND
                REFERENCED_TABLE_NAME IS NOT NULL;
            """
            cursor.execute(fk_query)
            schema['foreign_keys'] = cursor.fetchall()
    except pymysql.Error as e:
        logger.error(f"Error fetching schema for table `{table_name}`: {e}")
        raise
    return schema

def main():
    mysql_conn = None
    neo4j_driver = None
    try:
        mysql_conn = get_mysql_connection()
        neo4j_driver = get_neo4j_driver()

        # Disable Neo4j constraints for faster import if they exist
        # It's generally better to create them after migration for performance
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            # Drop existing constraints for a clean migration
            logger.info("Dropping existing Neo4j constraints...")
            result = session.run("SHOW CONSTRAINTS")
            for record in result:
                constraint_name = record["name"]
                try:
                    session.run(f"DROP CONSTRAINT `{constraint_name}`")
                    logger.info(f"Dropped constraint: {constraint_name}")
                except Exception as e:
                    logger.warning(f"Could not drop constraint {constraint_name}: {e}")
            logger.info("Finished dropping existing Neo4j constraints.")


        # Get all table names from MySQL
        tables_to_migrate = []
        with mysql_conn.cursor() as cursor:
            cursor.execute("SHOW TABLES")
            tables = cursor.fetchall()
            for table_dict in tables:
                table_name = list(table_dict.values())[0]
                tables_to_migrate.append(table_name)
        logger.info(f"Found tables to migrate: {tables_to_migrate}")

        # Phase 1: Node Creation (Multithreaded and Batched)
        logger.info("--- Starting Node Creation Phase ---")
        for table_name in tables_to_migrate:
            migrate_table_to_nodes(mysql_conn, neo4j_driver, table_name)
        logger.info("--- Node Creation Phase Completed ---")

        # Phase 2: Relationship Creation (using apoc.periodic.iterate)
        logger.info("--- Starting Relationship Creation Phase ---")
        for table_name in tables_to_migrate:
            schema = get_table_schema(mysql_conn, table_name)
            for fk in schema.get('foreign_keys', []):
                column_name = fk['COLUMN_NAME']
                referenced_table = fk['REFERENCED_TABLE_NAME']
                referenced_column = fk['REFERENCED_COLUMN_NAME']
                relationship_type = f"HAS_{column_name.upper().replace('ID', '')}" # Example: FK_customerId -> HAS_CUSTOMER

                # Ensure that the referenced table has been migrated as a node label
                if referenced_table.capitalize() in [t.capitalize() for t in tables_to_migrate]:
                    create_relationships_apoc(
                        neo4j_driver,
                        table_name.capitalize(),  # Source node label
                        'id',                     # Source node ID column (assuming 'id' for all tables)
                        referenced_table.capitalize(), # Target node label
                        referenced_column,        # Target node ID column
                        relationship_type,
                        column_name               # Column in source table that holds the FK value
                    )
                else:
                    logger.warning(f"Skipping relationship {relationship_type} from {table_name} to {referenced_table} "
                                   f"as {referenced_table} was not found as a migrated table.")
        logger.info("--- Relationship Creation Phase Completed ---")

        # Phase 3: Create Constraints (after all nodes are created for faster performance)
        logger.info("--- Creating Neo4j Constraints ---")
        with neo4j_driver.session(database=NEO4J_DATABASE) as session:
            for table_name in tables_to_migrate:
                label = table_name.capitalize()
                try:
                    # Assuming 'id' is the primary key in all MySQL tables and unique in Neo4j
                    session.run(f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.id IS UNIQUE")
                    logger.info(f"Created uniqueness constraint on {label}.id")
                except Exception as e:
                    logger.error(f"Error creating constraint on {label}.id: {e}")
        logger.info("--- Neo4j Constraint Creation Completed ---")

        logger.info("Database migration completed successfully!")

    except Exception as e:
        logger.critical(f"Migration failed: {e}")
    finally:
        if mysql_conn:
            mysql_conn.close()
            logger.info("MySQL connection closed.")
        if neo4j_driver:
            neo4j_driver.close()
            logger.info("Neo4j driver closed.")

if __name__ == "__main__":
    main()

2025-07-16 16:25:22,649 - MainThread - INFO - Successfully connected to MySQL database.
2025-07-16 16:25:24,731 - MainThread - INFO - Successfully connected to Neo4j database.
2025-07-16 16:25:24,733 - MainThread - INFO - Dropping existing Neo4j constraints...
2025-07-16 16:25:24,743 - MainThread - INFO - Finished dropping existing Neo4j constraints.
2025-07-16 16:25:24,764 - MainThread - INFO - Found tables to migrate: ['categories', 'countries', 'coupon_usages', 'coupons', 'customers', 'food_courts', 'grouped_texts', 'inventory_items', 'inventory_transactions', 'locales', 'menu_option_choice_to_menu_options', 'menu_option_choices', 'menu_option_to_menus', 'menu_options', 'menu_to_inventory_items', 'menus', 'order_items', 'orders', 'payment_option_enums', 'payments', 'refund_items', 'refunds', 'store_attributes', 'store_meta_advert_insights', 'store_meta_adverts', 'store_tables', 'stores', 'translated_texts']
2025-07-16 16:25:24,765 - MainThread - INFO - --- Starting Node Creation Pha

2025-07-16 16:25:27,337 - MainThread - ERROR - An unexpected error occurred during node migration for table `orders`: 'BoltDriver' object has no attribute 'execute_write'
2025-07-16 16:25:27,347 - MainThread - INFO - Starting migration for table: payment_option_enums to nodes with label: Payment_option_enums
2025-07-16 16:25:27,354 - MainThread - ERROR - MySQL error during migration of table `payment_option_enums`: (1054, "Unknown column 'deletedAt' in 'where clause'")
2025-07-16 16:25:27,355 - MainThread - INFO - Starting migration for table: payments to nodes with label: Payments
2025-07-16 16:25:27,365 - MainThread - INFO - Found 33614 active records in MySQL table `payments`.
2025-07-16 16:25:27,654 - MainThread - ERROR - An unexpected error occurred during node migration for table `payments`: 'BoltDriver' object has no attribute 'execute_write'
2025-07-16 16:25:27,660 - MainThread - INFO - Starting migration for table: refund_items to nodes with label: Refund_items
2025-07-16 16:2

2025-07-16 16:25:29,714 - MainThread - INFO - Successfully created relationships of type :HAS_GROUPEDTEXT.
2025-07-16 16:25:29,721 - MainThread - INFO - Creating relationships of type :HAS_MENU between Menu_to_inventory_items and Menus...
2025-07-16 16:25:29,789 - MainThread - INFO - Successfully created relationships of type :HAS_MENU.
2025-07-16 16:25:29,789 - MainThread - INFO - Creating relationships of type :HAS_INVENTORYITEM between Menu_to_inventory_items and Inventory_items...
2025-07-16 16:25:29,807 - MainThread - INFO - Successfully created relationships of type :HAS_INVENTORYITEM.
2025-07-16 16:25:29,816 - MainThread - INFO - Creating relationships of type :HAS_CATEGORY between Menus and Categories...
2025-07-16 16:25:29,914 - MainThread - INFO - Successfully created relationships of type :HAS_CATEGORY.
2025-07-16 16:25:29,915 - MainThread - INFO - Creating relationships of type :HAS_GROUPEDTEXT between Menus and Grouped_texts...
2025-07-16 16:25:29,932 - MainThread - INFO -

2025-07-16 16:25:32,718 - MainThread - INFO - Created uniqueness constraint on Refund_items.id
2025-07-16 16:25:32,796 - MainThread - INFO - Created uniqueness constraint on Refunds.id
2025-07-16 16:25:32,878 - MainThread - INFO - Created uniqueness constraint on Store_attributes.id
2025-07-16 16:25:32,966 - MainThread - INFO - Created uniqueness constraint on Store_meta_advert_insights.id
2025-07-16 16:25:33,050 - MainThread - INFO - Created uniqueness constraint on Store_meta_adverts.id
2025-07-16 16:25:33,121 - MainThread - INFO - Created uniqueness constraint on Store_tables.id
2025-07-16 16:25:33,204 - MainThread - INFO - Created uniqueness constraint on Stores.id
2025-07-16 16:25:33,284 - MainThread - INFO - Created uniqueness constraint on Translated_texts.id
2025-07-16 16:25:33,290 - MainThread - INFO - --- Neo4j Constraint Creation Completed ---
2025-07-16 16:25:33,292 - MainThread - INFO - Database migration completed successfully!
2025-07-16 16:25:33,294 - MainThread - INFO 